# 02. Prophet — Manufacturer Level

제조업체 단위 Prophet 모델 실험 노트북입니다.

- 월별 제조업체 판매량을 Prophet으로 예측
- MAPE 105% 수준으로 예측 불가 판정
- 제조업체 단위 접근의 한계를 정량적으로 확인

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import prophet

/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
final = pd.read_csv('../data/final.csv', encoding='utf-8-sig')
final

,판매월,제조업체,상품명,판매수량,대분류,중분류,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
1,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,2.0,면류.라면류,면류,22,0,0,101.20,105.5
2,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,1.0,면류.라면류,면류,22,0,0,101.20,105.5
3,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,2.0,면류.라면류,라면류,22,0,0,101.20,105.5
4,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
...,...,...,...,...,...,...,...,...,...,...,...
80578,2024-02,NaN,NaN,NaN,NaN,NaN,2,0,12,116.32,91.0
80579,2024-03,NaN,NaN,NaN,NaN,NaN,5,0,6,116.62,94.5
80580,2024-04,NaN,NaN,NaN,NaN,NaN,0,0,0,116.59,94.0
80581,2024-05,NaN,NaN,NaN,NaN,NaN,0,0,21,116.53,94.6


---

### 독립변수 조정
- 제조업체별 예측 ( ~제조업체별 판매횟수~ or 판매수량)
- ~중분류별 예측 (라면류, 면류, 기타면류)~
- 코로나 데이터 유무 (default : 제외하고, 위에서 성능 좋았던 경우에 추가)
- ~휴가 관련 데이터 추가~

---
### 모델 조정
- SARIMA 기반 예측
- 트리기반 모델 앙상블 (시계열 + 회귀 가능한 모델 search)

---
### 파라미터 조정
- 

---

## 제조업체별 예측

In [3]:
final_1 = final.copy()
final_1.drop(columns=['상품명', '대분류', '중분류'], inplace=True)

In [4]:
final_1

,판매월,제조업체,판매수량,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455.0,1.0,22,0,0,101.20,105.5
1,2021-01,88030600.0,2.0,22,0,0,101.20,105.5
2,2021-01,88030600.0,1.0,22,0,0,101.20,105.5
3,2021-01,88010455.0,2.0,22,0,0,101.20,105.5
4,2021-01,88010455.0,1.0,22,0,0,101.20,105.5
...,...,...,...,...,...,...,...,...
80578,2024-02,NaN,NaN,2,0,12,116.32,91.0
80579,2024-03,NaN,NaN,5,0,6,116.62,94.5
80580,2024-04,NaN,NaN,0,0,0,116.59,94.0
80581,2024-05,NaN,NaN,0,0,21,116.53,94.6


---

In [5]:
# 2024년 데이터 삭제
final_1['판매월'] = pd.to_datetime(final_1['판매월'], format='%Y-%m')
final_1 = final_1[final_1['판매월'].dt.year != 2024]

In [6]:
final_1.isnull().sum()

0

In [7]:
# 월별, 제조업체별 판매수량 합산
grouped_data = final_1.groupby(['판매월', '제조업체'], as_index=False).agg({
    '판매수량': 'sum',
    '한파': 'first',
    '폭염': 'first',
    '호우': 'first',
    '생활물가지수': 'first',
    '음식료품_계절조정지수': 'first'
})

grouped_data

,판매월,제조업체,판매수량,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01-01,88010249.0,4.0,22,0,0,101.20,105.5
1,2021-01-01,88010430.0,3719.0,22,0,0,101.20,105.5
2,2021-01-01,88010438.0,127.0,22,0,0,101.20,105.5
3,2021-01-01,88010453.0,450.0,22,0,0,101.20,105.5
4,2021-01-01,88010454.0,229.0,22,0,0,101.20,105.5
...,...,...,...,...,...,...,...,...
629,2023-12-01,88012772.0,3.0,14,0,2,114.84,94.6
630,2023-12-01,88023490.0,3.0,14,0,2,114.84,94.6
631,2023-12-01,88030600.0,559.0,14,0,2,114.84,94.6
632,2023-12-01,88092968.0,3.0,14,0,2,114.84,94.6


In [8]:
# 월별 제조업체의 unique 값과 월별 데이터 수 계산
monthly_unique_manufacturers = grouped_data.groupby('판매월')['제조업체'].nunique()
monthly_data_counts = grouped_data.groupby('판매월').size()

# 결과 비교
comparison = pd.DataFrame({
    'Unique 제조업체 수': monthly_unique_manufacturers,
    '총 데이터 수': monthly_data_counts
})

# 동일성 확인
comparison['동일한지 확인'] = comparison['Unique 제조업체 수'] == comparison['총 데이터 수']

# 결과 확인
comparison

,Unique 제조업체 수,총 데이터 수,동일한지 확인
판매월,,,
2021-01-01,13,13,True
2021-02-01,16,16,True
2021-03-01,15,15,True
2021-04-01,16,16,True
2021-05-01,17,17,True
2021-06-01,17,17,True
2021-07-01,17,17,True
2021-08-01,18,18,True
2021-09-01,16,16,True


In [21]:
# 각 제조업체의 예측 MAPE 결과 저장 리스트
monthly_mape_results = []

# 제조업체 리스트 가져오기
manufacturers = grouped_data['제조업체'].unique()

In [22]:
len(manufacturers)

23

In [23]:
manufacturers

array([88010249., 88010430., 88010438., 88010453., 88010454., 88010455.,
       88010456., 88010520., 88010731., 88010732., 88011285., 88030600.,
       88093891., 88012770., 99999999., 88092968., 88010527., 88010072.,
       88010399., 88010457., 88096952., 88023490., 88010733.])

In [13]:
# 2023-09기준으로 데이터 분리
grouped_data['판매월'] = pd.to_datetime(grouped_data['판매월'])
train = grouped_data[(grouped_data['판매월'] >= '2021-01-01') & (grouped_data['판매월'] <= '2023-08-31')]
test = grouped_data[(grouped_data['판매월'] >= '2023-09-01') & (grouped_data['판매월'] <= '2023-12-31')]

In [27]:
train['판매월'] = pd.to_datetime(train['판매월'])
test['판매월'] = pd.to_datetime(test['판매월'])

manufacturers = train['제조업체'].unique()

predictions = pd.DataFrame()

forecast_periods = pd.date_range(start='2023-09-01', end='2023-12-01', freq='MS')

/tmp/ipykernel_2235607/3462236410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['판매월'] = pd.to_datetime(train['판매월'])
/tmp/ipykernel_2235607/3462236410.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['판매월'] = pd.to_datetime(test['판매월'])


In [29]:
# 제조사별로 예측 수행
for manufacturer in manufacturers:
    # 해당 제조사의 훈련 데이터 선택
    manufacturer_data = train[train['제조업체'] == manufacturer]
    
    # 데이터 개수 확인
    if manufacturer_data.shape[0] < 2:
        print(f"{manufacturer} 데이터가 충분하지 않습니다. (데이터 수: {manufacturer_data.shape[0]})")
        continue
    
    # Prophet에 필요한 컬럼 이름 변경
    manufacturer_data = manufacturer_data.rename(columns={'판매월': 'ds', '판매수량': 'y'})
    
    # 모델 학습
    model = Prophet()
    
    # 외생변수 추가
    for col in ['한파', '폭염', '호우', '생활물가지수', '음식료품_계절조정지수']:
        model.add_regressor(col)
    
    model.fit(manufacturer_data)
    
    # test 데이터에서 외생변수 추출
    test_data = test[test['제조업체'] == manufacturer].copy()
    
    # 예측할 월 필터링
    test_data = test_data[test_data['판매월'].isin(forecast_periods)]
    
    if test_data.empty:
        print(f"{manufacturer}에 대한 예측할 데이터가 없습니다.")
        continue  # 예측할 데이터가 없으면 다음 제조사로 넘어감
    
    # 예측할 데이터프레임 구성
    test_data = test_data.rename(columns={'판매월': 'ds'})
    
    # 예측할 외생변수 추가
    # test 데이터는 이미 외생변수가 포함되어 있으므로 필요한 칼럼 선택
    forecast = model.predict(test_data[['ds'] + ['한파', '폭염', '호우', '생활물가지수', '음식료품_계절조정지수']])
    
    # 예측 결과에서 필요한 컬럼 선택 및 제조사 추가
    forecast = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    forecast['제조업체'] = manufacturer
    
    # 예측 결과를 predictions 데이터프레임에 추가
    predictions = pd.concat([predictions, forecast], ignore_index=True)

# 결과 확인
predictions

18:00:21 - cmdstanpy - INFO - Chain [1] start processing
18:00:21 - cmdstanpy - INFO - Chain [1] done processing
18:00:21 - cmdstanpy - INFO - Chain [1] start processing
18:00:21 - cmdstanpy - INFO - Chain [1] done processing
18:00:21 - cmdstanpy - INFO - Chain [1] start processing
18:00:21 - cmdstanpy - INFO - Chain [1] done processing
18:00:21 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1] done processing
18:00:22 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1] done processing
18:00:22 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1] done processing
18:00:22 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1] done processing
18:00:22 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1] done processing
18:00:22 - cmdstanpy - INFO - Chain [1] start processing
18:00:22 - cmdstanpy - INFO - Chain [1]

88010520.0에 대한 예측할 데이터가 없습니다.


18:00:23 - cmdstanpy - INFO - Chain [1] start processing
18:00:23 - cmdstanpy - INFO - Chain [1] done processing
18:00:23 - cmdstanpy - INFO - Chain [1] start processing
18:00:23 - cmdstanpy - INFO - Chain [1] done processing
18:00:23 - cmdstanpy - INFO - Chain [1] start processing
18:00:23 - cmdstanpy - INFO - Chain [1] done processing
18:00:23 - cmdstanpy - INFO - Chain [1] start processing
18:00:23 - cmdstanpy - INFO - Chain [1] done processing
18:00:23 - cmdstanpy - INFO - Chain [1] start processing


88093891.0에 대한 예측할 데이터가 없습니다.


18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing


...

88010399.0에 대한 예측할 데이터가 없습니다.
88012998.0에 대한 예측할 데이터가 없습니다.


18:00:28 - cmdstanpy - INFO - Chain [1] start processing
18:00:28 - cmdstanpy - INFO - Chain [1] done processing
18:00:28 - cmdstanpy - INFO - Chain [1] start processing
18:00:28 - cmdstanpy - INFO - Chain [1] done processing
18:00:28 - cmdstanpy - INFO - Chain [1] start processing


69728925.0에 대한 예측할 데이터가 없습니다.


18:00:30 - cmdstanpy - INFO - Chain [1] done processing
18:00:30 - cmdstanpy - INFO - Chain [1] start processing
18:00:30 - cmdstanpy - INFO - Chain [1] done processing
18:00:30 - cmdstanpy - INFO - Chain [1] start processing
18:00:30 - cmdstanpy - INFO - Chain [1] done processing
18:00:30 - cmdstanpy - INFO - Chain [1] start processing
18:00:30 - cmdstanpy - INFO - Chain [1] done processing


88010450.0 데이터가 충분하지 않습니다. (데이터 수: 1)
88010451.0 데이터가 충분하지 않습니다. (데이터 수: 1)
           ds         yhat   yhat_lower   yhat_upper        제조업체
0  2023-09-01    28.746802    27.448771    29.941547  88010249.0
1  2023-11-01     7.255611     5.962895     8.490813  88010249.0
2  2023-12-01    28.353286    27.030731    29.609155  88010249.0
3  2023-09-01  4407.793363  4335.893801  4483.285113  88010430.0
4  2023-10-01  2313.428472  2240.275293  2380.953180  88010430.0
..        ...          ...          ...          ...         ...
88 2023-12-01   -27.754803   -47.177734    -9.671605  88023490.0
89 2023-09-01    31.212903    24.592540    38.067624  88010733.0
90 2023-09-01   -11.876410   -11.876410   -11.876410  88012772.0
91 2023-11-01   314.902424   314.902423   314.902425  88012772.0
92 2023-12-01   252.795370   252.795368   252.795372  88012772.0

[93 rows x 5 columns]


In [31]:
predictions_sorted = predictions.sort_values(by=['ds', '제조업체']).reset_index(drop=True)
predictions_sorted

,ds,yhat,yhat_lower,yhat_upper,제조업체
0,2023-09-01,28.746802,27.448771,29.941547,88010249.0
1,2023-09-01,28.746802,27.495034,30.031145,88010249.0
2,2023-09-01,4407.793363,4335.893801,4483.285113,88010430.0
3,2023-09-01,4407.793363,4337.360492,4478.537036,88010430.0
4,2023-09-01,18.391513,-0.087178,36.958370,88010438.0
...,...,...,...,...,...
88,2023-12-01,252.795370,252.795368,252.795372,88012772.0
89,2023-12-01,-27.754803,-47.177734,-9.671605,88023490.0
90,2023-12-01,408.081550,217.743015,595.281268,88030600.0
91,2023-12-01,25.124342,22.042433,28.054937,88092968.0


In [36]:
# 월별로 yhat 합치기
predictions_sorted['year_month'] = predictions_sorted['ds'].dt.to_period('M')
monthly_sum = predictions_sorted.groupby('year_month')['yhat'].sum().reset_index()
monthly_sum['year_month'] = monthly_sum['year_month'].dt.to_timestamp()
monthly_sum

,year_month,yhat
0,2023-09-01,17408.174008
1,2023-10-01,9814.040358
2,2023-11-01,14441.964072
3,2023-12-01,14741.997196


In [35]:
# 월별로 판매수량 합치기
test['year_month'] = test['판매월'].dt.to_period('M')
monthly_sales_sum = test.groupby('year_month')['판매수량'].sum().reset_index()
monthly_sales_sum['year_month'] = monthly_sales_sum['year_month'].dt.to_timestamp()
monthly_sales_sum

/tmp/ipykernel_2235607/3372773759.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['year_month'] = test['판매월'].dt.to_period('M')


,year_month,판매수량
0,2023-09-01,6598.0
1,2023-10-01,7522.0
2,2023-11-01,6795.0
3,2023-12-01,6848.0


In [34]:
# MAPE 계산
comparison = pd.merge(monthly_sum, monthly_sales_sum, left_on='year_month', right_on='year_month', suffixes=('_pred', '_actual'))
comparison['mape'] = abs((comparison['yhat'] - comparison['판매수량']) / comparison['판매수량']) * 100
mean_mape = comparison['mape'].mean()
print(f'MAPE: {mean_mape:.2f}%')

MAPE: 105.53%


---

In [50]:
import pandas as pd
from prophet import Prophet
import numpy as np

# 데이터 로드
# final_1 = pd.read_csv('path_to_final_1.csv')  # 데이터 불러오기

# 판매월을 datetime 형식으로 변환
final_1['판매월'] = pd.to_datetime(final_1['판매월'], format='%Y-%m')

# 제조업체별로 그룹화하여 데이터 집계
grouped_data = final_1.groupby(['판매월', '제조업체'])['판매수량'].sum().reset_index()

# 결측치 0으로 대체
grouped_data['판매수량'].fillna(0, inplace=True)

# 각 제조업체의 예측 MAPE 결과 저장 리스트
monthly_mape_results = []

# 제조업체 리스트 가져오기
manufacturers = grouped_data['제조업체'].unique()

# 예측 결과를 저장할 리스트
forecast_results = []

/tmp/ipykernel_2234389/145319158.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  grouped_data['판매수량'].fillna(0, inplace=True)


In [51]:
for manufacturer in manufacturers:
    # 제조업체별 데이터 선택
    manufacturer_data = grouped_data[grouped_data['제조업체'] == manufacturer]

    # 데이터가 충분한지 확인
    if len(manufacturer_data) < 12:  # 최소 12개월의 데이터 필요
        print(f"{manufacturer} 데이터가 부족하여 건너뜁니다.")
        continue

    # 예측할 데이터 준비
    model_data = manufacturer_data.rename(columns={'판매월': 'ds', '판매수량': 'y'})
    
    # Prophet 모델 생성 및 학습
    model = Prophet()

    # 모델 학습
    model.fit(model_data)

    # 미래 예측을 위한 데이터 생성
    future = model.make_future_dataframe(periods=4, freq='M')  # 4개월 예측

    # 예측
    forecast = model.predict(future)

    # 예측 결과를 리스트에 추가
    for _, row in forecast.iterrows():
        forecast_results.append({
            '판매월': row['ds'],
            '제조업체': manufacturer,
            '예측판매수량': row['yhat']
        })

17:24:53 - cmdstanpy - INFO - Chain [1] start processing
17:24:53 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:53 - cmdstanpy - INFO - Chain [1] start processing
17:24:53 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:53 - cmdstanpy - INFO - Chain [1] start processing
17:24:53 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:53 - cmdstanpy - INFO - C

88012773.0 데이터가 부족하여 건너뜁니다.


17:24:55 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:55 - cmdstanpy - INFO - Chain [1] start processing
17:24:55 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:55 - cmdstanpy - INFO - Chain [1] start processing
17:24:56 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:56 - cmdstanpy - INFO - Chain [1] start processing
17:24:56 - cmdstanpy - INFO - C

88012998.0 데이터가 부족하여 건너뜁니다.


17:24:56 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:56 - cmdstanpy - INFO - Chain [1] start processing
17:24:56 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(
17:24:56 - cmdstanpy - INFO - Chain [1] start processing
17:24:56 - cmdstanpy - INFO - Chain [1] done processing
/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/prophet/forecaster.py:1854: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


69728925.0 데이터가 부족하여 건너뜁니다.
88010070.0 데이터가 부족하여 건너뜁니다.
88012772.0 데이터가 부족하여 건너뜁니다.
88010450.0 데이터가 부족하여 건너뜁니다.
88010451.0 데이터가 부족하여 건너뜁니다.


In [52]:
# 예측 결과를 데이터프레임으로 변환
forecast_df = pd.DataFrame(forecast_results)

# 실제 판매 수량과 예측 결과를 합치기
combined_data = forecast_df.merge(
    grouped_data,
    left_on=['판매월', '제조업체'],
    right_on=['판매월', '제조업체'],
    how='left'
)

# MAPE 계산
combined_data['absolute_error'] = np.abs(combined_data['판매수량'] - combined_data['예측판매수량'])
combined_data['mape'] = (combined_data['absolute_error'] / (combined_data['판매수량'] + 1e-10)) * 100  # 작은 값 추가하여 0으로 나누는 것을 방지

# 제조업체별로 MAPE 계산
mape_results = combined_data.groupby('제조업체').agg(
    평균_MAPE=('mape', 'mean'),
    총_예측판매수량=('예측판매수량', 'sum'),
    총_판매수량=('판매수량', 'sum')
).reset_index()

# 결과 DataFrame으로 변환
mape_df = pd.DataFrame(mape_results)

In [53]:
# MAPE 결과 출력
mape_df

,제조업체,평균_MAPE,총_예측판매수량,총_판매수량
0,88010072.0,159.944057,143.501862,108.0
1,88010249.0,31.399323,291.437317,265.0
2,88010399.0,47.297595,204.591338,182.0
3,88010430.0,8.418500,118566.117635,106999.0
4,88010438.0,26.320794,4335.122028,3838.0
5,88010453.0,15.077240,6540.628278,6270.0
6,88010454.0,31.793626,14504.736311,13272.0
7,88010455.0,11.299270,77836.322784,69525.0
8,88010456.0,31.801762,1485.012949,1352.0
9,88010457.0,31.302347,450.921856,406.0


---

In [41]:
monthly_manufacturer_count = final_1.groupby('판매월')['제조업체'].nunique().reset_index()
monthly_manufacturer_count.columns = ['판매월', '제조업체_개수']
monthly_manufacturer_count
#총 nunique 23개

,판매월,제조업체_개수
0,2021-01-01,13
1,2021-02-01,16
2,2021-03-01,15
3,2021-04-01,16
4,2021-05-01,17
5,2021-06-01,17
6,2021-07-01,17
7,2021-08-01,18
8,2021-09-01,16
9,2021-10-01,17


In [40]:
# '판매월' 별 각각의 '제조업체' 별 데이터 개수
monthly_manufacturer_count = final_1.groupby(['판매월', '제조업체']).size().reset_index(name='데이터_개수')
monthly_manufacturer_count

,판매월,제조업체,데이터_개수
0,2021-01-01,88010249.0,2
1,2021-01-01,88010430.0,1026
2,2021-01-01,88010438.0,62
3,2021-01-01,88010453.0,81
4,2021-01-01,88010454.0,67
...,...,...,...
629,2023-12-01,88012772.0,1
630,2023-12-01,88023490.0,1
631,2023-12-01,88030600.0,119
632,2023-12-01,88092968.0,3


---

## 결론

**1. SARIMAX 모델 사용 불가**
    
    1) 계속 발생한 에러코드
        : MAXLOG SHOULD BE < NOBS
    2) 여기서 nobs는 manufacturer_monthly['판매수량']의 길이 (maxlag:최대시차)
    3) 데이터가 너무 적어서 학습 불가

---
**2. prophet 모델로도 학습 불가**
    
    1) 모델이 학습하는 건 월별 제조업체별 데이터 (두 번째)
    2) 그 값이 너무 적음 >> 학습이 불가능한 case 多
    3) 예측: 월별로 제조업체별 판매수량을 예측해서 그걸 월별로 다시 합치는 형태
    4) 결과물은 월별 판매량을 예측한 것으로 나오지만, 결국 월별 제조업체별로 쪼개는 건 변함없기에 데이터 규모가 너무 작음
    5) 그래서 mape가 말도 안되게 크게 나온다

---    
**3) 제조업체별 데이터 예측은 불가능**